# Part 1 — Experiment 3: Preprocessing Technique 2 (Log Transform + Unsharp Masking) — Brain Tumor MRI

Completes the 3-experiment Part 1 protocol on **Brain Tumor MRI** (matching `0_Baseline_BrainTumor.ipynb` and `0_Baseline_pre_BrainTumor.ipynb`):
- **Experiment 1 (Baseline):** Frozen DINOv2 ViT-B/14 Linear Probe + `[5th, 95th]` percentile clip (`0_Baseline_BrainTumor.ipynb`)
- **Experiment 2 (Preprocessing 1):** CLAHE + Morphological Opening (`0_Baseline_pre_BrainTumor.ipynb`)
- **Experiment 3 (Preprocessing 2 — This Notebook):** **Logarithmic Intensity Transformation + Gaussian Unsharp Masking Sharpening**

### CV Curriculum Basis (Weeks 2–5)
- **Log Transform (`c * ln(1 + r)`):** Expands low-intensity MRI gray/white-matter boundaries to expose subtle infiltrative tumor margins.
- **Unsharp Masking:** Sharpens tumor capsule contours (particularly Meningioma dural margins) prior to frozen DINOv2 feature extraction.

In [1]:
"""
Colab environment setup: clone repository, install dependencies, mount Google Drive.
"""
import os
import subprocess

if os.path.basename(os.getcwd()) != "DinoV2_PEFT":
    if not os.path.exists("DinoV2_PEFT"):
        subprocess.run(["git", "clone", "https://github.com/RudhanLodhi/DinoV2_PEFT.git"], check=True)
    os.chdir("DinoV2_PEFT")

subprocess.run(["pip", "install", "-r", "requirements.txt", "opencv-python-headless", "requests", "-q"], check=True)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    CHECKPOINT_DIR = "/content/drive/MyDrive/dinov2_peft/exp3_pre2_log_unsharp_brain_tumor"
except ImportError:
    CHECKPOINT_DIR = "outputs/exp3_pre2_log_unsharp_brain_tumor"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

Mounted at /content/drive
Checkpoint directory: /content/drive/MyDrive/dinov2_peft/exp3_pre2_log_unsharp_brain_tumor


## 1. Download MRI Slices, Apply Log + Unsharp Preprocessing & Build DataLoaders

In [2]:
"""
Download Brain Tumor MRI images (if needed) and apply Preprocessing 2 (Percentile Clip + Log + Unsharp).
"""
from io import BytesIO
import json
from pathlib import Path
import random
import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import requests
import torch
import torchvision.transforms as T
from tqdm.auto import tqdm
from src.data import build_brain_tumor_dataloaders
from src.data.transforms import MRIPercentileClip
from src.utils.config import load_config

NDJSON_PATH = Path("cheng-brain-tumor-mri-detection.ndjson")
IMAGE_ROOT = Path("data/brain_tumor/images")
IMAGE_ROOT.mkdir(parents=True, exist_ok=True)

records = []
with open(NDJSON_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        r = json.loads(line)
        if r.get("type") == "image" and r.get("url"):
            records.append(r)

for r in tqdm(records, desc="Checking/Downloading MRI images"):
    dest = IMAGE_ROOT / r["file"]
    if not dest.exists():
        resp = requests.get(r["url"], timeout=15)
        resp.raise_for_status()
        Image.open(BytesIO(resp.content)).convert("RGB").save(dest)

class MRILogUnsharpTransform:
    """Applies [5th, 95th] percentile clip -> Logarithmic intensity transform -> Gaussian Unsharp Masking."""
    def __init__(self, sharp_amount: float = 0.5) -> None:
        self.clip = MRIPercentileClip(5.0, 95.0)
        self.c = 255.0 / np.log1p(255.0)
        self.sharp_amount = sharp_amount

    def __call__(self, img: Image.Image) -> Image.Image:
        clipped = np.array(self.clip(img).convert("L"), dtype=np.float32)
        log_img = np.clip(self.c * np.log1p(clipped), 0, 255).astype(np.uint8)
        blurred = cv2.GaussianBlur(log_img, (5, 5), sigmaX=1.0)
        sharpened = cv2.addWeighted(log_img, 1.0 + self.sharp_amount, blurred, -self.sharp_amount, 0)
        return Image.fromarray(sharpened).convert("RGB")

cfg = load_config("configs/brain_tumor.yaml")
pre = cfg["preprocessing"]
pre2_tf = MRILogUnsharpTransform(sharp_amount=0.5)

train_tf = T.Compose([
    pre2_tf,
    T.RandomResizedCrop(pre["image_size"], scale=(0.75, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.ToTensor(),
    T.Normalize(mean=pre["normalize_mean"], std=pre["normalize_std"]),
])
val_tf = T.Compose([
    pre2_tf,
    T.Resize(int(pre["image_size"] * 1.14)),
    T.CenterCrop(pre["image_size"]),
    T.ToTensor(),
    T.Normalize(mean=pre["normalize_mean"], std=pre["normalize_std"]),
])

train_loader, val_loader, test_loader = build_brain_tumor_dataloaders(
    cfg, NDJSON_PATH, image_root=IMAGE_ROOT, train_transforms=train_tf, val_transforms=val_tf
)
CLASS_NAMES = ["Meningioma", "Glioma", "Pituitary"]
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")

Checking/Downloading MRI images:   0%|          | 0/3064 [00:00<?, ?it/s]

HTTPError: 403 Client Error: Forbidden for url: https://cdn.ul.run/eu/i/013d531c418e0ebb0d3e12a62beff619.jpg?Expires=1793937600&KeyName=key-v1&Signature=Vlw5_K6az3ZWFb2j5mncU6YQj7I

## 2. Train Frozen DINOv2 ViT-B/14 Linear Probe (Identical Settings to Exp 1 & Exp 2)

In [ ]:
"""
Train frozen DINOv2 ViT-B/14 linear probe on Brain Tumor MRI with Log + Unsharp preprocessing.
"""
import copy
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score
from torch.optim import SGD
from src.evaluation.metrics import compute_multiclass_metrics
from src.models import count_trainable_parameters, freeze_backbone, load_backbone, MultiClassHead
from src.training.losses import MultiClassCELoss

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
backbone, _ = load_backbone()
freeze_backbone(backbone)
head = MultiClassHead(in_features=768, num_classes=3)

class LinearProbeBT(nn.Module):
    def __init__(self, b: nn.Module, h: nn.Module) -> None:
        super().__init__()
        self.backbone = b
        self.head = h
    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        return self.head(self.backbone(pixel_values=pixel_values).last_hidden_state[:, 0, :])

model = LinearProbeBT(backbone, head).to(device)
print(f"Trainable params: {count_trainable_parameters(model)[0]:,}")

criterion = MultiClassCELoss()
optimizer = SGD(head.parameters(), lr=0.01, momentum=0.9, weight_decay=0.0)
epochs = 20
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
scaler = torch.amp.GradScaler(enabled=(device.type == "cuda"))

@torch.no_grad()
def evaluate_bt(m, loader):
    m.eval()
    logits_l, labels_l = [], []
    for b in loader:
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            out = m(b["pixel_values"].to(device, non_blocking=True))
        logits_l.append(out.float().cpu().numpy())
        labels_l.append(b["labels"].numpy())
    probs = torch.softmax(torch.from_numpy(np.concatenate(logits_l)), dim=-1).numpy()
    y = np.concatenate(labels_l).astype(int)
    preds = np.argmax(probs, axis=1)
    mc = compute_multiclass_metrics(y, probs, class_names=CLASS_NAMES)
    return {
        "mean_auroc": float(mc["mean_auroc"]),
        "accuracy": float(mc["accuracy"]),
        "macro_f1": float(mc["macro_f1"]),
        "macro_precision": float(precision_score(y, preds, average="macro", zero_division=0.0)),
        "macro_recall": float(recall_score(y, preds, average="macro", zero_division=0.0)),
        "confusion_matrix": confusion_matrix(y, preds).tolist(),
        "report": classification_report(y, preds, target_names=CLASS_NAMES, zero_division=0.0),
    }

history, best_auc, best_state = [], -1.0, None
for ep in range(1, epochs + 1):
    model.train()
    rl = 0.0
    for b in train_loader:
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            loss = criterion(model(b["pixel_values"].to(device)), b["labels"].to(device))
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        rl += loss.item()
    scheduler.step()
    vm = evaluate_bt(model, val_loader)
    history.append({"epoch": ep, "train_loss": rl / len(train_loader), "val_auroc": vm["mean_auroc"], "val_acc": vm["accuracy"]})
    print(f"Epoch [{ep:02d}/{epochs}] Loss: {rl/len(train_loader):.4f} | Val AUROC: {vm['mean_auroc']:.4f} | Val Acc: {vm['accuracy']:.4f}")
    if vm["mean_auroc"] > best_auc:
        best_auc = vm["mean_auroc"]
        best_state = copy.deepcopy(model.state_dict())

model.load_state_dict(best_state)
val_res = evaluate_bt(model, val_loader)
test_res = evaluate_bt(model, test_loader)
print("\nTest Classification Report:\n", test_res["report"])
print(f"Best Val AUROC: {val_res['mean_auroc']:.4f} | Test AUROC: {test_res['mean_auroc']:.4f} | Test Acc: {test_res['accuracy']:.4f}")
with open(f"{CHECKPOINT_DIR}/results.json", "w", encoding="utf-8") as f:
    json.dump({"val": val_res, "test": test_res, "history": history}, f, indent=2)

## 3. Part 1 Three-Experiment Comparison Table & Confusion Matrix (Brain Tumor MRI)

In [ ]:
"""
Render Part 1 Section 7 comparison table (Exp 1 Baseline vs Exp 2 CLAHE+Morph vs Exp 3 Log+Unsharp) and Confusion Matrix.
"""
import pandas as pd
import seaborn as sns

part1_bt_summary = pd.DataFrame([
    {"Experiment": "Exp 1: Baseline (0_Baseline_BrainTumor)", "Val AUROC": 0.9701, "Val Acc": 0.8633, "Base Paper Target": 0.9720},
    {"Experiment": "Exp 2: Preprocessing 1 — CLAHE + Morph", "Val AUROC": 0.9696, "Val Acc": 0.8618, "Base Paper Target": 0.9720},
    {"Experiment": "Exp 3: Preprocessing 2 — Log + Unsharp", "Val AUROC": round(val_res["mean_auroc"], 4), "Val Acc": round(val_res["accuracy"], 4), "Base Paper Target": 0.9720},
])
print(part1_bt_summary.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
hdf = pd.DataFrame(history)
sns.set_theme(style="whitegrid")
axes[0].plot(hdf["epoch"], hdf["val_auroc"], marker="s", color="darkorange", label="Exp 3 Val AUROC")
axes[0].axhline(0.9720, color="crimson", linestyle="--", label="Base Paper (0.9720)")
axes[0].set_title("Exp 3 (Log + Unsharp) Val AUROC — Brain Tumor MRI")
axes[0].legend()

cm = np.array(test_res["confusion_matrix"])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
axes[1].set_title("Exp 3 Held-Out Test Confusion Matrix")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Ground Truth")
plt.tight_layout()
plt.savefig(f"{CHECKPOINT_DIR}/exp3_brain_tumor_summary.png", dpi=200)
plt.show()